# 🔄 Interview Questions: ETL & Data Pipelines
## The Core of Data Engineering

### 🎯 Why ETL Defines Your Career as a Data Engineer

**ETL isn't just moving data—it's the backbone of every data platform.** Here's why:

1. **Foundation of All Analytics** - Without reliable ETL, everything downstream fails
2. **80% of DE Work** - Most data engineering time is spent building and maintaining pipelines
3. **Production Impact** - Broken pipelines = executives making decisions on stale data
4. **Scale Complexity** - ETL at scale (TB-PB) requires deep technical expertise
5. **Career Differentiator** - Senior engineers design resilient, self-healing pipelines

### 💡 What Separates Junior from Senior Data Engineers

| Junior Engineer | Senior Data Engineer |
|----------------|----------------------|
| "Let's do a full load every time" | "Design incremental with idempotency" |
| Writes brittle scripts that break in production | Designs fault-tolerant, self-healing pipelines |
| No error handling or monitoring | Comprehensive logging, alerting, and recovery |
| "Just run it manually if it fails" | Automated retry logic and dead-letter queues |
| Ignores data quality until issues arise | Validates at every stage with circuit breakers |

---

### 📊 Interview Question Coverage (25 Questions)

This module covers **7 critical ETL domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **ETL Design Patterns** | 5 | Architecture fundamentals |
| **Data Extraction Strategies** | 3 | Source system integration |
| **Transformation Best Practices** | 5 | Data quality and business logic |
| **Loading Strategies** | 3 | Target optimization |
| **Error Handling & Logging** | 3 | Production resilience |
| **Pipeline Orchestration** | 3 | Workflow management |
| **Data Lineage & Auditing** | 3 | Governance and compliance |

---

### 🎓 How to Master This Module

1. **Think idempotent** - Every pipeline should produce the same result when run multiple times
2. **Design for failure** - Assume everything will fail and plan accordingly
3. **Optimize for maintainability** - You'll spend more time debugging than writing
4. **Know your tools** - Delta Live Tables, Auto Loader, dbt, Airflow
5. **Monitor everything** - You can't fix what you can't see

### 🏆 Interview Success Tips

✅ **Draw pipeline architecture diagrams** on the whiteboard
✅ **Explain incremental load strategies** (CDC, watermarks, delta detection)
✅ **Mention Databricks tools** (Delta Live Tables, Auto Loader, Unity Catalog)
✅ **Discuss data quality** (validation, monitoring, alerting)
✅ **Show production mindset** (error handling, retry logic, idempotency)

⚠️ **Red flags that fail interviews:**
- Suggesting full loads for large tables
- No error handling or monitoring strategy
- Not understanding idempotency
- Ignoring data quality validation
- Can't explain how to recover from failures

---

**Ready to master ETL & data pipelines? Let's dive in!** 🔥

## 📌 Section 1: ETL Design Patterns (5 Questions)

Design patterns form the foundation of scalable, maintainable data pipelines.

### ❓ Question 1: Design an ETL Pipeline for Daily Customer Updates

**Real-World Interview Question:**
> "You have a MySQL database with 50 million customer records that changes daily. Design an ETL pipeline to sync these updates to your data warehouse (Databricks). Explain your extraction strategy, transformation logic, and loading approach. How do you handle new customers, updates, and deletes?"

### ✅ Answer 1: Daily Customer Update Pipeline Design

#### **Architecture Overview**

```
                  MySQL Source
                  (50M customers)
                       |
                       | CDC / Incremental Query
                       v
              [ EXTRACT LAYER ]
              Auto Loader / JDBC
                       |
                       v
                 Bronze Table
              (Raw Landing Zone)
                       |
                       | Data Quality Checks
                       v
             [ TRANSFORM LAYER ]
            (Dedupe, Enrich, Validate)
                       |
                       v
                 Silver Table
              (Cleaned & Enriched)
                       |
                       | Business Logic
                       v
                  Gold Table
              (Analytics-Ready)
```

#### **1. Extraction Strategy (3 Options)**

**Option A: Change Data Capture (CDC) - BEST**

```sql
-- Use Debezium or native MySQL binlog streaming
-- Databricks Auto Loader reads CDC events from Kafka/cloud storage

CREATE OR REPLACE STREAMING LIVE TABLE bronze_customers_raw
COMMENT "Raw CDC events from MySQL"
AS SELECT 
  *,
  _metadata.file_path,
  _metadata.file_modification_time
FROM cloud_files(
  "s3://my-bucket/cdc/customers/",
  "json",
  map("cloudFiles.inferColumnTypes", "true",
      "cloudFiles.schemaLocation", "s3://my-bucket/schemas/customers")
);
```

**Pros:**
- ✅ Real-time / near real-time
- ✅ Captures all operations (INSERT, UPDATE, DELETE)
- ✅ Minimal source database load
- ✅ Low latency

**Cons:**
- ❌ Complex setup (Debezium, Kafka)
- ❌ Source DB must support CDC

---

**Option B: Incremental Query with Watermark - GOOD**

```python
# PySpark incremental load using modified_timestamp
from pyspark.sql.functions import col, max as spark_max

# Get last successful run timestamp
last_watermark = spark.sql("""
  SELECT MAX(modified_timestamp) 
  FROM silver.customers
""").collect()[0][0] or '1970-01-01'

# Extract only changed records
incremental_df = (
  spark.read
    .format("jdbc")
    .option("url", mysql_url)
    .option("dbtable", "(SELECT * FROM customers WHERE modified_timestamp > '{}') AS q".format(last_watermark))
    .option("numPartitions", "10")
    .load()
)

incremental_df.write.mode("append").saveAsTable("bronze.customers_raw")
```

**Pros:**
- ✅ Simple to implement
- ✅ Works with any database
- ✅ Reliable watermark tracking

**Cons:**
- ❌ Requires modified_timestamp column
- ❌ Can't detect hard deletes
- ❌ Higher DB load than CDC

---

**Option C: Full Load with Delta Comparison - FALLBACK**

```python
# Full load to staging, then compare
current_df = spark.read.format("jdbc").load()
previous_df = spark.table("silver.customers")

# Detect changes
changed_records = current_df.subtract(previous_df)
```

**Use ONLY when:**
- Table is small (<10M rows)
- No modified_timestamp available
- CDC not supported

---

#### **2. Transformation Logic (Bronze → Silver → Gold)**

**Bronze Layer: Raw Landing**

```sql
-- Raw CDC events or incremental extracts
CREATE OR REPLACE TABLE bronze.customers_raw (
  customer_id STRING,
  email STRING,
  first_name STRING,
  last_name STRING,
  phone STRING,
  address STRING,
  city STRING,
  state STRING,
  zip_code STRING,
  account_status STRING,
  modified_timestamp TIMESTAMP,
  operation STRING,  -- INSERT, UPDATE, DELETE (from CDC)
  _ingestion_timestamp TIMESTAMP,
  _source_file STRING
);
```

**Silver Layer: Cleaned & Deduplicated**

```sql
CREATE OR REPLACE LIVE TABLE silver_customers
COMMENT "Cleaned customer data with deduplication"
AS
WITH deduplicated AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id 
      ORDER BY modified_timestamp DESC, _ingestion_timestamp DESC
    ) AS row_num
  FROM LIVE.bronze_customers_raw
  WHERE 
    -- Data quality checks
    customer_id IS NOT NULL
    AND email IS NOT NULL
    AND email RLIKE '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Z|a-z]{2,}$'
)
SELECT 
  customer_id,
  LOWER(TRIM(email)) AS email,
  INITCAP(TRIM(first_name)) AS first_name,
  INITCAP(TRIM(last_name)) AS last_name,
  REGEXP_REPLACE(phone, '[^0-9]', '') AS phone,
  UPPER(TRIM(state)) AS state,
  LPAD(zip_code, 5, '0') AS zip_code,
  account_status,
  modified_timestamp,
  operation,
  CURRENT_TIMESTAMP() AS processed_timestamp
FROM deduplicated
WHERE row_num = 1  -- Keep latest version
  AND operation != 'DELETE';  -- Filter soft deletes
```

**Gold Layer: Business Logic**

```sql
CREATE OR REPLACE LIVE TABLE gold_customers_active
COMMENT "Active customers with enrichment"
AS
SELECT 
  c.customer_id,
  c.email,
  CONCAT(c.first_name, ' ', c.last_name) AS full_name,
  c.state,
  -- Enrichment: Add customer segment
  CASE 
    WHEN o.lifetime_value > 10000 THEN 'VIP'
    WHEN o.lifetime_value > 1000 THEN 'Premium'
    ELSE 'Standard'
  END AS customer_segment,
  o.total_orders,
  o.lifetime_value,
  c.modified_timestamp
FROM LIVE.silver_customers c
LEFT JOIN LIVE.gold_order_summary o ON c.customer_id = o.customer_id
WHERE c.account_status = 'ACTIVE';
```

---

#### **3. Loading Strategy: MERGE (Upsert)**

```sql
-- Handle INSERT, UPDATE, DELETE in single operation
MERGE INTO gold.customers AS target
USING (
  SELECT * FROM silver.customers_incremental
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.operation = 'DELETE' THEN
  DELETE
WHEN MATCHED THEN
  UPDATE SET 
    target.email = source.email,
    target.first_name = source.first_name,
    target.last_name = source.last_name,
    target.phone = source.phone,
    target.state = source.state,
    target.modified_timestamp = source.modified_timestamp,
    target.updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED AND source.operation != 'DELETE' THEN
  INSERT (customer_id, email, first_name, last_name, phone, state, modified_timestamp, created_at)
  VALUES (source.customer_id, source.email, source.first_name, source.last_name, 
          source.phone, source.state, source.modified_timestamp, CURRENT_TIMESTAMP());
```

---

#### **4. Handling Deletes (3 Strategies)**

**Hard Delete:**
```sql
-- Physically remove rows (not recommended for analytics)
DELETE FROM gold.customers 
WHERE customer_id IN (SELECT customer_id FROM source WHERE operation = 'DELETE');
```

**Soft Delete (Recommended):**
```sql
-- Add is_deleted flag
ALTER TABLE gold.customers ADD COLUMN is_deleted BOOLEAN DEFAULT FALSE;

UPDATE gold.customers 
SET is_deleted = TRUE, deleted_at = CURRENT_TIMESTAMP()
WHERE customer_id IN (SELECT customer_id FROM source WHERE operation = 'DELETE');

-- Queries filter deleted records
SELECT * FROM gold.customers WHERE is_deleted = FALSE;
```

**SCD Type 2 (Full History):**
```sql
-- Track all versions over time
CREATE TABLE gold.customers_history (
  customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id STRING,
  email STRING,
  effective_date DATE,
  expiration_date DATE,
  is_current BOOLEAN
);

-- Mark current version as expired, insert new version
```

---

#### **5. Idempotency & Error Handling**

```python
# Idempotent pipeline execution
def run_customer_etl(run_date):
    try:
        # 1. Extract with watermark
        watermark = get_last_watermark(run_date)
        extract_customers(watermark, run_date)
        
        # 2. Transform (Delta Live Tables handles deduplication)
        transform_customers(run_date)
        
        # 3. Load (MERGE is idempotent)
        load_customers(run_date)
        
        # 4. Update watermark
        update_watermark(run_date)
        
        # 5. Data quality checks
        validate_row_counts()
        validate_no_nulls()
        
    except Exception as e:
        log_error(run_date, str(e))
        send_alert("Customer ETL Failed", e)
        raise  # Fail pipeline for retry

# Run pipeline (can run multiple times safely)
run_customer_etl('2024-01-15')
```

---

#### **Interview Follow-Up Questions & Answers**

**Q: "How do you handle late-arriving data?"**
A: "Use event_timestamp (when the change occurred) separate from processing_timestamp. Allow a late-arrival window (e.g., 3 days) and reprocess if needed."

**Q: "How do you monitor pipeline health?"**
A: 
- Row count validation (source vs target)
- Data freshness alerts (modified_timestamp lag)
- Schema drift detection
- Data quality metrics (null rates, duplicate rates)

**Q: "How do you optimize for 50M records?"**
A:
- Partition by date (modified_date)
- Z-ORDER on customer_id
- Use Auto Loader for incremental ingestion
- Run OPTIMIZE on target tables

**Q: "How do you handle schema changes?"**
A: "Use schema evolution in Delta Lake + Auto Loader's schema inference. Store schema versions in a catalog."

---

#### **Production Considerations**

✅ **Monitoring:**
- Track extraction lag (source modified_timestamp vs ingestion_timestamp)
- Alert on row count anomalies (>10% variance)
- Monitor pipeline duration (baseline: 10 minutes)

✅ **Performance:**
- Partition target table by modified_date
- Z-ORDER by customer_id for point lookups
- Use Liquid Clustering for evolving access patterns

✅ **Data Quality:**
- Validate email format
- Check for duplicate customer_ids
- Ensure required fields are not null
- Compare source and target row counts

### ❓ Question 2: Full Load vs Incremental Load Trade-offs

**Critical Interview Question:**
> "Explain the difference between full load and incremental load ETL patterns. When would you use each? What are the trade-offs in terms of performance, complexity, and data consistency?"

### ✅ Answer 2: Full Load vs Incremental Load

#### **Full Load Pattern**

**Definition:** Extract the ENTIRE dataset from source on every run, replace target table.

```sql
-- Full load pattern
CREATE OR REPLACE TABLE target.customers AS
SELECT * FROM source.customers;
```

**Architecture:**
```
Source (10M rows)
       |
       | Read ALL rows
       v
   Staging Table
       |
       | TRUNCATE + INSERT
       v
   Target Table
```

**Pros:**
- ✅ **Simple** - No tracking logic needed
- ✅ **Self-healing** - Automatically fixes past errors
- ✅ **No watermarks** - No state management
- ✅ **Catches deletes** - Deleted rows disappear naturally

**Cons:**
- ❌ **Slow** - Processes ALL data every time
- ❌ **High source load** - Full table scans
- ❌ **Network intensive** - Transfers entire dataset
- ❌ **Loses history** - Can't track changes over time

**When to use:**
- ✅ Small tables (<1M rows)
- ✅ Static reference data (country codes, product categories)
- ✅ No historical tracking needed
- ✅ Source system doesn't support incremental

---

#### **Incremental Load Pattern**

**Definition:** Extract only CHANGED records since last run, merge into target.

```sql
-- Incremental load with watermark
SELECT * 
FROM source.customers
WHERE modified_timestamp > (SELECT MAX(modified_timestamp) FROM target.customers);

-- Merge into target
MERGE INTO target.customers AS t
USING incremental_data AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

**Architecture:**
```
Source (10M rows)
       |
       | Read ONLY changed rows (1K rows)
       v
   Incremental Extract
       |
       | MERGE (Upsert)
       v
   Target Table (preserves history)
```

**Pros:**
- ✅ **Fast** - Processes only changed data
- ✅ **Low source load** - Efficient queries
- ✅ **Scalable** - Works with billions of rows
- ✅ **Can track history** - With SCD patterns

**Cons:**
- ❌ **Complex** - Watermark management
- ❌ **State management** - Track last run timestamp
- ❌ **Hard delete handling** - Requires soft delete flags
- ❌ **Can drift** - If watermark tracking fails

**When to use:**
- ✅ Large tables (>10M rows)
- ✅ Frequent updates (hourly, real-time)
- ✅ Historical tracking needed (SCD Type 2)
- ✅ Source has modified_timestamp

---

#### **Trade-off Comparison**

| Factor | Full Load | Incremental Load |
|--------|-----------|------------------|
| **Performance** | Slow (⏱️ hours) | Fast (⏱️ minutes) |
| **Source Load** | High 🔴 | Low 🟢 |
| **Complexity** | Simple 🟢 | Complex 🟡 |
| **Scalability** | Poor (>10M rows) | Excellent (billions) |
| **Data Consistency** | Always consistent | Risk of drift |
| **Handle Deletes** | Automatic | Requires soft delete |
| **Historical Tracking** | No | Yes (SCD Type 2) |
| **Network Usage** | High | Low |
| **Best For** | Small, static tables | Large, frequently changing |

---

#### **Hybrid Pattern: Full Refresh + Incremental**

**Best practice:** Use incremental daily, full refresh weekly/monthly.

```python
# Hybrid ETL pattern
def run_etl(date, force_full_load=False):
    if force_full_load or is_first_day_of_month(date):
        # Monthly full refresh for data consistency
        full_load_customers()
        log("Full load executed")
    else:
        # Daily incremental for performance
        incremental_load_customers(date)
        log("Incremental load executed")
    
    # Validate consistency
    validate_row_counts()
```

**Benefits:**
- ✅ Performance of incremental (99% of time)
- ✅ Self-healing via periodic full refresh
- ✅ Catches watermark drift

---

#### **Incremental Load Strategies**

**1. Timestamp-Based (Most Common)**

```sql
-- Extract changes
SELECT * 
FROM source.customers
WHERE modified_timestamp > :last_watermark
  AND modified_timestamp <= :current_watermark;

-- Store watermark
INSERT INTO etl_metadata.watermarks VALUES
  ('customers', :current_watermark, CURRENT_TIMESTAMP());
```

**Requirements:**
- ✅ modified_timestamp column exists
- ✅ Updated on every change (trigger/app logic)
- ✅ Indexed for performance

**2. Change Data Capture (CDC)**

```sql
-- Read CDC stream from Debezium/Kafka
CREATE OR REPLACE STREAMING LIVE TABLE customers_cdc
AS SELECT 
  op,  -- 'c' (create), 'u' (update), 'd' (delete)
  after.*,
  ts_ms AS cdc_timestamp
FROM cloud_files("s3://cdc-bucket/customers/", "json");
```

**Benefits:**
- ✅ Captures ALL changes (insert, update, delete)
- ✅ Near real-time
- ✅ No impact on source DB

**3. Sequence/Version Number**

```sql
-- For systems without timestamp
SELECT * 
FROM source.customers
WHERE version_number > (SELECT MAX(version_number) FROM target.customers);
```

---

#### **Handling Edge Cases**

**Late-Arriving Data:**

```python
# Allow 3-day late arrival window
last_watermark = current_date - timedelta(days=3)

# Reprocess last 3 days to catch late data
incremental_df = extract_changes(last_watermark, current_date)
```

**Hard Deletes:**

```sql
-- Option 1: Soft delete flag (recommended)
ALTER TABLE customers ADD COLUMN is_deleted BOOLEAN DEFAULT FALSE;

-- Option 2: Full diff (expensive)
WITH target_ids AS (SELECT customer_id FROM target.customers),
     source_ids AS (SELECT customer_id FROM source.customers)
SELECT t.customer_id 
FROM target_ids t
LEFT JOIN source_ids s ON t.customer_id = s.customer_id
WHERE s.customer_id IS NULL;  -- Deleted in source
```

**Clock Skew:**

```python
# Add buffer for clock skew between systems
watermark = last_run_time - timedelta(minutes=5)
```

---

#### **Performance Optimization**

**Full Load:**
```sql
-- Partition source query
SELECT * FROM source.customers WHERE MOD(customer_id, 10) = :partition_id;

-- Use bulk copy utilities
COPY INTO target.customers
FROM 's3://bucket/full-extract/'
FILEFORMAT = PARQUET;
```

**Incremental Load:**
```sql
-- Partition by date
CREATE TABLE target.customers
PARTITIONED BY (modified_date DATE);

-- Only scan recent partitions
SELECT * FROM source.customers
WHERE modified_date >= CURRENT_DATE - INTERVAL 7 DAYS;
```

---

#### **Interview Answer Template**

1. **Start with the question:** "Is this a small reference table or a large transactional table?"
2. **Explain trade-offs:** "Full load is simpler but doesn't scale. Incremental is complex but performant."
3. **Mention hybrid approach:** "In production, I'd use incremental daily with weekly full refresh."
4. **Ask about requirements:** "Do you need historical tracking? How are deletes handled?"
5. **Show you understand production:** "I'd add watermark management, validation checks, and monitoring."

### ❓ Question 3: How to Implement Idempotent ETL Pipelines?

**Senior-Level Interview Question:**
> "Explain what idempotency means in data pipelines. Why is it critical for production systems? Walk me through how you would make a pipeline idempotent—provide SQL examples for MERGE operations and discuss partition handling."

### ✅ Answer 3: Idempotent Pipeline Implementation

#### **What is Idempotency?**

**Definition:** Running the same pipeline multiple times produces the same result.

```
f(x) = y
f(f(x)) = y
f(f(f(x))) = y
```

**Why Critical:**
- 🚀 **Safe retries** - Can rerun failed pipelines without duplicates
- 🚀 **Backfilling** - Reprocess historical data without side effects
- 🚀 **Late data** - Handle late-arriving records correctly
- 🚀 **Production reliability** - Eliminates manual cleanup after failures

**Non-Idempotent (BAD):**
```sql
-- Running twice creates duplicates!
INSERT INTO target.sales 
SELECT * FROM source.sales WHERE date = '2024-01-15';

-- Result after 2 runs: 2x the data 🚨
```

**Idempotent (GOOD):**
```sql
-- Running twice produces same result
MERGE INTO target.sales AS t
USING (SELECT * FROM source.sales WHERE date = '2024-01-15') AS s
ON t.sale_id = s.sale_id AND t.date = s.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- Result after 2 runs: correct data ✅
```

---

#### **Strategy 1: MERGE (Upsert) Pattern**

**Basic MERGE:**

```sql
-- Idempotent upsert using MERGE
MERGE INTO target.customers AS target
USING (
  SELECT 
    customer_id,
    email,
    name,
    modified_timestamp
  FROM source.customers
  WHERE date = :run_date  -- Partition pruning
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN 
  UPDATE SET 
    target.email = source.email,
    target.name = source.name,
    target.modified_timestamp = source.modified_timestamp,
    target.updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (customer_id, email, name, modified_timestamp, created_at)
  VALUES (source.customer_id, source.email, source.name, 
          source.modified_timestamp, CURRENT_TIMESTAMP());
```

**Key Points:**
- ✅ Uses natural key (customer_id) for matching
- ✅ UPDATE for existing records
- ✅ INSERT for new records
- ✅ Running twice = same final state

---

**Handling Partition Overwrites (Databricks-Specific):**

```python
# Idempotent partition overwrite
from pyspark.sql import functions as F

# Read source data for specific date
source_df = (
    spark.read
        .format("delta")
        .load("s3://source/sales")
        .filter(F.col("date") == "2024-01-15")
)

# Overwrite ONLY the specific partition
(
    source_df
        .write
        .format("delta")
        .mode("overwrite")  # Overwrites only the partition!
        .option("replaceWhere", "date = '2024-01-15'")  # Idempotent!
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("date")
        .saveAsTable("target.sales")
)

# Running twice: same result (no duplicates)
```

**Critical:** Use `replaceWhere` to overwrite ONLY specific partition!

---

#### **Strategy 2: Deduplication Before Load**

```sql
-- Deduplicate source data before MERGE
WITH deduplicated_source AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id 
      ORDER BY modified_timestamp DESC, _ingestion_timestamp DESC
    ) AS row_num
  FROM staging.customers
  WHERE date = :run_date
)
MERGE INTO target.customers AS t
USING (
  SELECT * FROM deduplicated_source WHERE row_num = 1
) AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

**Benefits:**
- ✅ Handles duplicates in source
- ✅ Keeps latest version
- ✅ Idempotent even if source has duplicates

---

#### **Strategy 3: Truncate + Insert Pattern (Simpler for Small Tables)**

```sql
-- For dimension tables or small datasets
BEGIN TRANSACTION;

-- Delete partition
DELETE FROM target.customers WHERE date = :run_date;

-- Insert new data
INSERT INTO target.customers
SELECT * FROM source.customers WHERE date = :run_date;

COMMIT;
```

**Use when:**
- ✅ Small tables (<10M rows per partition)
- ✅ Full partition refresh
- ✅ No need to preserve history

---

#### **Strategy 4: Delta Lake MERGE with Conditions**

**Update only if source is newer:**

```sql
MERGE INTO target.customers AS t
USING source.customers AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.modified_timestamp > t.modified_timestamp THEN
  UPDATE SET 
    t.email = s.email,
    t.name = s.name,
    t.modified_timestamp = s.modified_timestamp
WHEN NOT MATCHED THEN
  INSERT (customer_id, email, name, modified_timestamp)
  VALUES (s.customer_id, s.email, s.name, s.modified_timestamp);
```

**Benefits:**
- ✅ Prevents overwriting newer data with older data
- ✅ Handles out-of-order processing
- ✅ Idempotent even with late-arriving data

---

#### **Strategy 5: Atomic File Writes**

```python
# Write to temporary location, then atomic rename
temp_path = f"s3://bucket/temp/{run_id}/"
final_path = f"s3://bucket/production/date={run_date}/"

try:
    # Write to temp location
    df.write.format("delta").save(temp_path)
    
    # Atomic move (idempotent)
    dbutils.fs.rm(final_path, recurse=True)  # Remove old
    dbutils.fs.mv(temp_path, final_path)     # Atomic rename
    
except Exception as e:
    # Cleanup temp files
    dbutils.fs.rm(temp_path, recurse=True)
    raise
```

**Benefits:**
- ✅ Atomic operation (all or nothing)
- ✅ No partial writes on failure
- ✅ Can retry safely

---

#### **Idempotency Checklist**

**✅ Data Layer:**
- [ ] Use MERGE instead of INSERT
- [ ] Use replaceWhere for partition overwrites
- [ ] Deduplicate before load
- [ ] Use natural keys for matching
- [ ] Handle out-of-order data with timestamps

**✅ Metadata Layer:**
```sql
-- Track pipeline runs idempotently
CREATE TABLE etl_metadata.pipeline_runs (
  pipeline_name STRING,
  run_date DATE,
  run_id STRING,
  status STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  PRIMARY KEY (pipeline_name, run_date)
);

-- Idempotent upsert
MERGE INTO etl_metadata.pipeline_runs AS t
USING (SELECT :pipeline_name AS pipeline_name, :run_date AS run_date) AS s
ON t.pipeline_name = s.pipeline_name AND t.run_date = s.run_date
WHEN MATCHED THEN 
  UPDATE SET status = 'RUNNING', start_time = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (pipeline_name, run_date, run_id, status, start_time)
  VALUES (s.pipeline_name, s.run_date, :run_id, 'RUNNING', CURRENT_TIMESTAMP());
```

**✅ Processing Logic:**
```python
def process_sales_data(run_date):
    """
    Idempotent sales processing pipeline
    """
    # 1. Check if already processed (optional optimization)
    if is_already_processed(run_date):
        logger.info(f"{run_date} already processed")
        return  # Skip, already done
    
    # 2. Extract (idempotent - same query every time)
    df = extract_sales(run_date)
    
    # 3. Transform (deterministic - same input = same output)
    transformed_df = transform_sales(df)
    
    # 4. Load (MERGE - idempotent)
    load_sales(transformed_df, run_date)
    
    # 5. Mark complete (MERGE - idempotent)
    mark_processed(run_date)
```

---

#### **Common Pitfalls (Non-Idempotent)**

**❌ Auto-Increment IDs:**
```sql
-- BAD: Auto-increment creates different IDs each run
INSERT INTO customers (id, name) 
VALUES (nextval('customer_id_seq'), 'Alice');

-- GOOD: Use natural key or UUID
INSERT INTO customers (customer_id, name) 
VALUES ('CUST-12345', 'Alice')
ON CONFLICT (customer_id) DO UPDATE SET name = EXCLUDED.name;
```

**❌ CURRENT_TIMESTAMP() in Transformations:**
```sql
-- BAD: Different timestamp each run
SELECT 
    customer_id,
    CURRENT_TIMESTAMP() AS processed_at  -- Changes!
FROM source.customers;

-- GOOD: Use event timestamp or pass as parameter
SELECT 
    customer_id,
    :run_timestamp AS processed_at  -- Same value
FROM source.customers;
```

**❌ Append-Only Without Deduplication:**
```sql
-- BAD: Creates duplicates
INSERT INTO target.events
SELECT * FROM source.events WHERE date = :run_date;

-- GOOD: MERGE with event_id
MERGE INTO target.events USING source.events ...
```

---

#### **Testing Idempotency**

```python
import pytest

def test_idempotency():
    run_date = '2024-01-15'
    
    # Run pipeline first time
    run_pipeline(run_date)
    result_1 = spark.table("target.sales").filter(f"date = '{run_date}'").count()
    
    # Run pipeline second time (should be identical)
    run_pipeline(run_date)
    result_2 = spark.table("target.sales").filter(f"date = '{run_date}'").count()
    
    # Assert same result
    assert result_1 == result_2, "Pipeline is not idempotent!"
    
    # Verify no duplicates
    duplicates = spark.sql(f"""
        SELECT sale_id, COUNT(*) AS cnt
        FROM target.sales
        WHERE date = '{run_date}'
        GROUP BY sale_id
        HAVING COUNT(*) > 1
    """).count()
    
    assert duplicates == 0, "Found duplicate records!"
```

---

#### **Production Example: Idempotent Delta Live Tables Pipeline**

```python
import dlt
from pyspark.sql import functions as F

@dlt.table(
    name="silver_customers",
    comment="Idempotent customer processing"
)
def silver_customers():
    """
    Idempotent transformation using Delta Live Tables
    """
    return (
        dlt.read("bronze_customers")
        # Deduplicate (idempotent)
        .withColumn(
            "row_num",
            F.row_number().over(
                Window.partitionBy("customer_id")
                .orderBy(F.desc("modified_timestamp"))
            )
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

# Delta Live Tables automatically handles MERGE for you!
# Re-running the pipeline produces the same result.
```

---

#### **Interview Answer Template**

1. **Define:** "Idempotency means running the pipeline N times produces the same result as running it once."
2. **Why important:** "Critical for production—enables safe retries, backfills, and recovery from failures."
3. **Implementation:** "Use MERGE instead of INSERT, partition overwrites with replaceWhere, and deduplication."
4. **Show code:** Provide SQL MERGE example with natural key matching.
5. **Mention pitfalls:** "Avoid auto-increment IDs, CURRENT_TIMESTAMP in transformations, and append-only without deduplication."
6. **Testing:** "Always test by running twice and comparing row counts + checking for duplicates."

### ❓ Question 4: Batch vs Streaming ETL — When to Use Each?

**Architecture Interview Question:**
> "Compare batch and streaming ETL approaches. When would you choose one over the other? How does the Lambda architecture address the trade-offs between batch and streaming?"

### ✅ Answer 4: Batch vs Streaming ETL

#### **Batch ETL (Traditional)**

**Definition:** Process data in large chunks on a schedule (hourly, daily, weekly).

**Architecture:**
```
Source DB --> [Wait for batch window] --> ETL Job --> Target Warehouse
             (accumulate changes)        (process all)

Example: Daily 2 AM job processes yesterday's transactions
```

**Pros:**
- ✅ **Simple** - Easier to reason about, debug, and maintain
- ✅ **Cost-effective** - Cluster spins up, processes, shuts down
- ✅ **High throughput** - Optimized for bulk processing
- ✅ **Mature tooling** - Airflow, dbt, Databricks Jobs
- ✅ **Easy reprocessing** - Backfills are straightforward

**Cons:**
- ❌ **High latency** - Hours/days old data
- ❌ **Resource spikes** - Large compute during batch window
- ❌ **Limited real-time** - Can't support live dashboards

**When to use:**
- ✅ Analytics and reporting (daily/hourly reports)
- ✅ Data warehouse loads
- ✅ ML model training
- ✅ Historical analysis
- ✅ Cost-sensitive workloads

**Example:**
```python
# Batch ETL in Databricks
def daily_batch_etl():
    # Process yesterday's data
    yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    
    df = spark.read.format("delta").load("bronze.transactions") \
        .filter(f"date = '{yesterday}'")
    
    # Transform
    transformed = df.groupBy("customer_id").agg(
        sum("amount").alias("total_spent"),
        count("*").alias("transaction_count")
    )
    
    # Load (MERGE for idempotency)
    transformed.write.format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"date = '{yesterday}'") \
        .save("gold.customer_daily_summary")
```

---

#### **Streaming ETL (Real-Time)**

**Definition:** Process data continuously as it arrives (seconds/milliseconds latency).

**Architecture:**
```
Source --> [Stream] --> Continuous Processing --> Target (updated in real-time)
           (Kafka)      (Structured Streaming)       (Delta Lake)

Example: Process transactions as they occur
```

**Pros:**
- ✅ **Low latency** - Seconds to minutes
- ✅ **Real-time insights** - Live dashboards, alerts
- ✅ **Smooth resource usage** - Constant load (no spikes)
- ✅ **Event-driven** - React to changes immediately

**Cons:**
- ❌ **Complex** - Harder to debug, requires state management
- ❌ **Higher cost** - Cluster runs 24/7
- ❌ **Limited tooling** - Fewer mature orchestration tools
- ❌ **Harder reprocessing** - Backfills more complex

**When to use:**
- ✅ Real-time fraud detection
- ✅ Live dashboards
- ✅ IoT sensor data
- ✅ Financial trading systems
- ✅ Operational monitoring

**Example:**
```python
# Streaming ETL in Databricks
from pyspark.sql.functions import window, sum, count

# Read stream from Kafka
stream_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "broker:9092")
        .option("subscribe", "transactions")
        .load()
)

# Parse and transform
transformed = (
    stream_df
        .selectExpr("CAST(value AS STRING) as json")
        .select(from_json("json", schema).alias("data"))
        .select("data.*")
        # Windowed aggregation
        .withWatermark("timestamp", "10 minutes")
        .groupBy(
            window("timestamp", "5 minutes"),
            "customer_id"
        )
        .agg(
            sum("amount").alias("total_spent"),
            count("*").alias("transaction_count")
        )
)

# Write stream to Delta Lake
query = (
    transformed.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", "/checkpoints/transactions")
        .table("gold.customer_realtime_summary")
)
```

---

#### **Comparison**

| Factor | Batch | Streaming |
|--------|-------|----------|
| **Latency** | Hours-Days | Seconds-Minutes |
| **Complexity** | Low 🟢 | High 🔴 |
| **Cost** | Low (on-demand) | High (always-on) |
| **Throughput** | Very high | Moderate |
| **Debugging** | Easy | Hard |
| **Backfills** | Easy | Complex |
| **State Management** | None | Required |
| **Use Case** | Analytics, reports | Real-time alerts |

---

#### **Lambda Architecture (Hybrid)**

**Definition:** Combine batch and streaming for best of both worlds.

**Architecture:**
```
                    Source Data
                        |
            +-----------+-----------+
            |                       |
       Batch Layer             Speed Layer
     (slow, accurate)        (fast, approximate)
            |                       |
    Batch Views               Real-time Views
            |                       |
            +----------+------------+
                       |
                  Serving Layer
               (merge both views)
```

**Example:**
- **Speed Layer:** Streaming job provides real-time counts (may have duplicates)
- **Batch Layer:** Daily batch job provides accurate counts (deduplicated)
- **Serving Layer:** Query merges both, preferring batch when available

**Benefits:**
- ✅ Real-time updates (speed layer)
- ✅ Accurate results (batch layer corrects errors)
- ✅ Fault tolerant (batch layer is source of truth)

**Drawbacks:**
- ❌ Two codebases to maintain
- ❌ Complex query layer

---

#### **Kappa Architecture (Streaming-Only)**

**Definition:** Everything is a stream. Batch is just bounded stream.

**Architecture:**
```
Source --> Kafka --> Streaming Processor --> Delta Lake
                           |
                   Single codebase!
                   (process everything as stream)
```

**Example in Databricks:**
```python
# Same code for batch and streaming!
def process_data(is_streaming=True):
    if is_streaming:
        df = spark.readStream.format("delta").load("source")
    else:
        df = spark.read.format("delta").load("source")
    
    # Same transformation code
    result = df.groupBy("customer_id").agg(sum("amount"))
    
    if is_streaming:
        result.writeStream.format("delta").start()
    else:
        result.write.format("delta").save()
```

**Benefits:**
- ✅ Single codebase (easier maintenance)
- ✅ Simpler architecture
- ✅ Batch = replay stream from beginning

---

#### **Decision Matrix**

**Choose Batch when:**
- ✅ Latency > 15 minutes is acceptable
- ✅ Data arrives in files (daily dumps)
- ✅ Cost is a concern
- ✅ Simple maintenance is priority

**Choose Streaming when:**
- ✅ Latency < 5 minutes required
- ✅ Data arrives continuously (Kafka, IoT)
- ✅ Real-time decisions needed
- ✅ Event-driven architecture

**Choose Hybrid (Lambda/Kappa) when:**
- ✅ Need real-time + accuracy
- ✅ Complex aggregations
- ✅ Late-arriving data is common

---

#### **Databricks-Specific Recommendations**

**Use Delta Live Tables for both:**
```python
import dlt

# Works for batch AND streaming!
@dlt.table
def silver_transactions():
    return dlt.read_stream("bronze_transactions")  # Streaming
    # Or: dlt.read("bronze_transactions")  # Batch

# Delta Live Tables handles the complexity!
```

**Use Auto Loader for incremental batch:**
```python
# Best of both: batch simplicity + streaming-like latency
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("s3://bucket/data/")
)
# Processes new files as they arrive (micro-batches)
```

---

#### **Interview Answer Template**

1. **Latency requirements:** "What's your latency SLA? >15 min = batch, <5 min = streaming."
2. **Data arrival pattern:** "Files or streams? Files = batch, Kafka = streaming."
3. **Complexity trade-off:** "Batch is simpler. Use streaming only if you NEED low latency."
4. **Mention hybrid:** "For critical systems, Lambda architecture provides both speed and accuracy."
5. **Databricks tools:** "Delta Live Tables abstracts batch vs streaming. Auto Loader gives you incremental batch."

### ❓ Question 5: How Do You Monitor and Alert on Pipeline Health?

**Production-Focused Interview Question:**
> "Your customer ETL pipeline runs nightly at 2 AM. How do you monitor its health? What metrics do you track? When should alerts fire? Walk me through your monitoring and alerting strategy."

### ✅ Answer 5: Pipeline Monitoring & Alerting

#### **Monitoring Framework (4 Layers)**

```
Layer 1: Data Quality Metrics
Layer 2: Pipeline Performance Metrics  
Layer 3: Infrastructure Metrics
Layer 4: Business Metrics
```

---

#### **Layer 1: Data Quality Monitoring**

**Row Count Validation:**
```sql
-- Compare source vs target row counts
CREATE OR REPLACE TABLE monitoring.data_quality_checks AS
WITH source_count AS (
  SELECT 'source' AS location, COUNT(*) AS row_count
  FROM source.customers WHERE date = :run_date
),
target_count AS (
  SELECT 'target' AS location, COUNT(*) AS row_count  
  FROM target.customers WHERE date = :run_date
)
SELECT 
  :run_date AS check_date,
  s.row_count AS source_rows,
  t.row_count AS target_rows,
  ABS(s.row_count - t.row_count) AS row_diff,
  CASE 
    WHEN ABS(s.row_count - t.row_count) / s.row_count > 0.05 THEN 'ALERT'
    ELSE 'OK'
  END AS status
FROM source_count s, target_count t;

-- Alert if >5% variance
```

**Null Rate Monitoring:**
```sql
SELECT 
  :run_date AS check_date,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS email_null_rate,
  SUM(CASE WHEN phone IS NULL THEN 1 ELSE 0 END) / COUNT(*) AS phone_null_rate
FROM target.customers
WHERE date = :run_date
HAVING email_null_rate > 0.01;  -- Alert if >1% nulls
```

**Duplicate Detection:**
```sql
SELECT 
  customer_id,
  COUNT(*) AS duplicate_count
FROM target.customers
WHERE date = :run_date
GROUP BY customer_id
HAVING COUNT(*) > 1;  -- Alert if any duplicates
```

---

#### **Layer 2: Pipeline Performance Metrics**

**Track in Metadata Table:**
```sql
CREATE TABLE monitoring.pipeline_metrics (
  pipeline_name STRING,
  run_date DATE,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  duration_minutes DOUBLE,
  rows_processed BIGINT,
  rows_inserted BIGINT,
  rows_updated BIGINT,
  rows_deleted BIGINT,
  status STRING,  -- SUCCESS, FAILED, RUNNING
  error_message STRING
);

-- Insert metrics after each run
INSERT INTO monitoring.pipeline_metrics VALUES (
  'customer_etl',
  CURRENT_DATE(),
  :start_time,
  CURRENT_TIMESTAMP(),
  TIMESTAMPDIFF(MINUTE, :start_time, CURRENT_TIMESTAMP()),
  :rows_processed,
  :rows_inserted,
  :rows_updated,
  :rows_deleted,
  'SUCCESS',
  NULL
);
```

**Duration Anomaly Detection:**
```sql
-- Alert if duration > 2x baseline
WITH baseline AS (
  SELECT AVG(duration_minutes) AS avg_duration
  FROM monitoring.pipeline_metrics
  WHERE pipeline_name = 'customer_etl'
    AND status = 'SUCCESS'
    AND run_date >= CURRENT_DATE() - INTERVAL 30 DAYS
),
today AS (
  SELECT duration_minutes
  FROM monitoring.pipeline_metrics
  WHERE pipeline_name = 'customer_etl'
    AND run_date = CURRENT_DATE()
)
SELECT 
  'Duration Alert' AS alert_type,
  t.duration_minutes AS current_duration,
  b.avg_duration AS baseline_duration
FROM today t, baseline b
WHERE t.duration_minutes > b.avg_duration * 2;
```

---

#### **Layer 3: Infrastructure Monitoring**

**Cluster Health:**
```python
# Monitor via Databricks API
import requests

def check_cluster_health(cluster_id):
    response = requests.get(
        f"{databricks_url}/api/2.0/clusters/get",
        headers={"Authorization": f"Bearer {token}"},
        json={"cluster_id": cluster_id}
    )
    cluster = response.json()
    
    # Check state
    if cluster['state'] != 'RUNNING':
        send_alert(f"Cluster {cluster_id} is {cluster['state']}")
    
    # Check autoscaling
    if cluster['num_workers'] >= cluster['autoscale']['max_workers']:
        send_alert(f"Cluster {cluster_id} at max capacity")
```

**Delta Lake Health:**
```sql
-- Monitor table size and file count
DESCRIBE DETAIL target.customers;

-- Alert if too many small files
SELECT 
  'Small Files Alert' AS alert_type,
  numFiles,
  sizeInBytes / numFiles AS avg_file_size_bytes
FROM (DESCRIBE DETAIL target.customers)
WHERE numFiles > 1000 AND sizeInBytes / numFiles < 10000000;  -- <10MB avg
```

---

#### **Layer 4: Business Metrics**

**Data Freshness:**
```sql
-- Alert if data is stale
SELECT 
  MAX(modified_timestamp) AS latest_data_timestamp,
  TIMESTAMPDIFF(HOUR, MAX(modified_timestamp), CURRENT_TIMESTAMP()) AS staleness_hours
FROM target.customers
HAVING staleness_hours > 36;  -- Alert if >36 hours old
```

**Business KPIs:**
```sql
-- Monitor daily transaction volume
WITH today_volume AS (
  SELECT COUNT(*) AS transaction_count
  FROM gold.transactions
  WHERE date = CURRENT_DATE()
),
baseline AS (
  SELECT AVG(cnt) AS avg_count, STDDEV(cnt) AS std_count
  FROM (
    SELECT date, COUNT(*) AS cnt
    FROM gold.transactions
    WHERE date >= CURRENT_DATE() - INTERVAL 30 DAYS
    GROUP BY date
  )
)
SELECT 
  'Transaction Volume Alert',
  t.transaction_count,
  b.avg_count AS baseline
FROM today_volume t, baseline b
WHERE t.transaction_count < b.avg_count - (2 * b.std_count);  -- 2 std devs below
```

---

#### **Alerting Strategy**

**Alert Severity Levels:**

| Severity | Condition | Action | Response Time |
|----------|-----------|--------|---------------|
| **CRITICAL** | Pipeline failed | Page on-call | Immediate |
| **HIGH** | Data quality failure | Email + Slack | <1 hour |
| **MEDIUM** | Performance anomaly | Slack notification | <4 hours |
| **LOW** | Warning threshold | Log only | Next day |

**Alert Implementation:**
```python
def send_alert(severity, message, metrics={}):
    """
    Multi-channel alerting
    """
    if severity == 'CRITICAL':
        # PagerDuty for on-call
        pagerduty.trigger_incident(message)
        # Slack
        slack.post_message('#data-alerts', f":rotating_light: CRITICAL: {message}")
        # Email
        send_email(team_email, f"CRITICAL ALERT: {message}")
    
    elif severity == 'HIGH':
        slack.post_message('#data-alerts', f":warning: HIGH: {message}")
        send_email(team_email, f"High Priority Alert: {message}")
    
    elif severity == 'MEDIUM':
        slack.post_message('#data-monitoring', f":yellow_circle: MEDIUM: {message}")
    
    else:  # LOW
        logger.warning(message)
    
    # Log all alerts
    log_alert(severity, message, metrics)
```

---

#### **Alert Rules for Customer ETL**

```python
# Alert configuration
ALERT_RULES = {
    'pipeline_failure': {
        'severity': 'CRITICAL',
        'message': 'Customer ETL pipeline failed',
        'channels': ['pagerduty', 'slack', 'email']
    },
    'row_count_variance': {
        'severity': 'HIGH',
        'threshold': 0.05,  # >5% variance
        'message': 'Row count variance exceeds 5%',
        'channels': ['slack', 'email']
    },
    'duration_anomaly': {
        'severity': 'MEDIUM',
        'threshold': 2.0,  # 2x baseline
        'message': 'Pipeline duration exceeds baseline',
        'channels': ['slack']
    },
    'data_freshness': {
        'severity': 'HIGH',
        'threshold_hours': 36,
        'message': 'Data staleness exceeds 36 hours',
        'channels': ['slack', 'email']
    },
    'null_rate': {
        'severity': 'MEDIUM',
        'threshold': 0.01,  # >1%
        'message': 'Null rate exceeds 1%',
        'channels': ['slack']
    },
    'duplicates_detected': {
        'severity': 'HIGH',
        'threshold': 0,
        'message': 'Duplicate records detected',
        'channels': ['slack', 'email']
    }
}
```

---

#### **Monitoring Dashboard (SQL)**

```sql
-- Daily pipeline health dashboard
CREATE OR REPLACE VIEW monitoring.pipeline_health_dashboard AS
SELECT 
  DATE(start_time) AS run_date,
  pipeline_name,
  status,
  duration_minutes,
  rows_processed,
  -- Data quality score
  CASE 
    WHEN status = 'SUCCESS' 
      AND rows_processed > 0 
      AND duration_minutes < 60 THEN 100
    WHEN status = 'SUCCESS' 
      AND rows_processed > 0 THEN 75
    WHEN status = 'FAILED' THEN 0
    ELSE 50
  END AS health_score,
  error_message
FROM monitoring.pipeline_metrics
WHERE start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY start_time DESC;
```

---

#### **Proactive Monitoring (Predict Failures)**

```python
# ML-based anomaly detection
from databricks import automl

# Train model on historical metrics
history = spark.table("monitoring.pipeline_metrics")

# Features: duration, rows_processed, day_of_week, hour
features = history.select(
    "duration_minutes",
    "rows_processed",
    dayofweek("start_time").alias("day_of_week"),
    hour("start_time").alias("hour"),
    (col("status") == "FAILED").cast("int").alias("failed")
)

# AutoML trains anomaly detection model
summary = automl.classify(features, target_col="failed")

# Predict if tonight's run will fail
tonight_features = ...
prediction = model.predict(tonight_features)

if prediction == 1:
    send_alert('MEDIUM', 'Tonight\'s pipeline run may fail based on recent trends')
```

---

#### **Interview Answer Template**

1. **4 Layers:** "Monitor data quality, pipeline performance, infrastructure, and business metrics."
2. **Data Quality:** "Row count validation, null rates, duplicates, schema drift."
3. **Performance:** "Duration, throughput, lag, resource usage."
4. **Alerting:** "Severity levels (critical/high/medium/low) with appropriate channels (PagerDuty/Slack/email)."
5. **Proactive:** "Anomaly detection using historical baselines and ML models."
6. **Tools:** "Databricks SQL for metrics, Delta Lake DESCRIBE DETAIL, custom Python monitoring."

## 📌 Section 2: Data Extraction Strategies (3 Questions)

Efficient data extraction is critical for pipeline performance and source system health.